# Arnowitt-Deser-Misner (ADM) and Baumgarte-Shapiro-Shibata-Nakamura (BSSN) Conversions

## Author: Zach Etienne

To run the whole notebook, click the `>>` toolbar button and choose
**Restart Kernel and Run All Cells...**.

This notebook follows the ADM-to-BSSN conversion pipeline in NRPy, inspects
the variables produced by the forward conversion, and checks that a toy ADM
data set can be converted to BSSN and reconstructed back to ADM fields.

**Notebook Status:** Validated from a fresh kernel.

**Validation Notes:** The final round-trip residuals compare every
reconstructed ADM metric, curvature, shift, and determinant component against
the trusted toy ADM input.

### Required Reading

- [ADM 3+1 Background](adm_3plus1_background.ipynb): ADM field names and
  physical roles.
- This notebook defines the additional BSSN tensor symbols and
  symmetric-component ordering before use.
- [Brown's covariant BSSN formulation](https://arxiv.org/abs/0902.3652).
- [Reference-metric BSSN conventions](https://arxiv.org/abs/1211.6632).
- [NRPy+/reference-metric BSSN implementation background](https://arxiv.org/abs/1712.07658).

### NRPy Source Code

These links point to upstream NRPy source for inspection. The notebook imports
the installed package and does not assume a local source checkout.

- [ADM_to_BSSN.py](https://github.com/nrpy/nrpy/blob/main/nrpy/equations/general_relativity/ADM_to_BSSN.py):
  forward ADM-to-BSSN conversion formulas.
- [BSSN_to_ADM.py](https://github.com/nrpy/nrpy/blob/main/nrpy/equations/general_relativity/BSSN_to_ADM.py):
  inverse BSSN-to-ADM reconstruction formulas.
- [indexedexp.py](https://github.com/nrpy/nrpy/blob/main/nrpy/indexedexp.py):
  tensor/list constructors and symmetric matrix helpers used in the toy check.
- [params.py](https://github.com/nrpy/nrpy/blob/main/nrpy/params.py):
  runtime parameter interface used to choose the conformal factor.

Navigation:
[Index](../index.ipynb) |
Previous: [ADM 3+1 Background](adm_3plus1_background.ipynb) |
Next: [Index](../index.ipynb)

# Table of Contents

1. [Notation and Terms](#Notation-and-Terms)
1. [Step 1: Map ADM fields to BSSN variables](#Step-1:-Map-ADM-fields-to-BSSN-variables)
1. [Step 2: Choose the conformal-factor variable](#Step-2:-Choose-the-conformal-factor-variable)
1. [Step 3: Convert Brill-Lindquist ADM data](#Step-3:-Convert-Brill-Lindquist-ADM-data)
1. [Validation Check: Brill-Lindquist BSSN residuals](#Validation-Check:-Brill-Lindquist-BSSN-residuals)
1. [Step 4: Build a toy ADM test case](#Step-4:-Build-a-toy-ADM-test-case)
1. [Step 5: Run the forward ADM-to-BSSN conversion](#Step-5:-Run-the-forward-ADM-to-BSSN-conversion)
1. [Step 6: Build the inverse substitution map](#Step-6:-Build-the-inverse-substitution-map)
1. [Validation Check: Reconstruct ADM fields from computed BSSN values](#Validation-Check:-Reconstruct-ADM-fields-from-computed-BSSN-values)
1. [What next?](#What-next?)

# Notation and Terms
### [Back to [top](#Table-of-Contents)]

| Name | Meaning in this notebook | Code name |
| --- | --- | --- |
| Dimension | all tensors here have three spatial components | `range(3)` |
| Index range | Latin spatial indices `i,j` run over `0,1,2` | list indices |
| Basis | all conversion checks in this notebook use Cartesian coordinates | `"Cartesian"` |
| Lower indices | covariant tensor components | `gammaDD[i][j]`, `KDD[i][j]` |
| Upper indices | contravariant vector components | `betaU[i]`, `vetU[i]` |
| Symmetric rank-2 tensor | full 3-by-3 tensor with six independent components | `gammaDD`, `KDD`, `hDD`, `aDD` |
| Contraction | repeated upper/lower index pairs are summed | `K = gamma^{ij} K_{ij}` |
| Reference metric | coordinate-system baseline geometry | `ghatDD` |
| Reference determinant | determinant of the reference metric | `detgammahat` |
| Reference rescaling factors | factors removed from components | `ReDD`, `ReU` |
| Conformal factor | removed spatial scale variable | `cf` |
| Metric correction | conformal metric minus the reference metric | `hDD` |
| Trace-free curvature | curvature with its trace removed | `aDD` |
| Curvature trace | contraction of `KDD` with the inverse metric | `trK` |
| Rescaled shift | shift divided by the reference scale factor | `vetU` |
| Residual | expression expected to simplify to zero | `sp.factor(...)` |

NRPy names independent symmetric rank-2 components in upper-triangular order:
`(00, 01, 02, 11, 12, 22)`. The next inspection cell verifies that ordering
before later cells use flattened names such as `hDD01` and `aDD12`.

The next cell imports standard-library output control, SymPy, NRPy's
indexed-expression helpers, the runtime parameter interface, and the
conversion classes used below.

In [1]:
import contextlib
import io

import sympy as sp

import nrpy.indexedexp as ixp
import nrpy.params as par
from nrpy.equations.general_relativity.ADM_to_BSSN import ADM_to_BSSN
from nrpy.equations.general_relativity.BSSN_to_ADM import BSSN_to_ADM
from nrpy.equations.general_relativity.InitialData_Cartesian import (
    InitialData_Cartesian,
)

Before using flattened symmetric tensor names, inspect the complete `hDD`
template. The repeated lower-triangular entries show symmetry, while the
independent order is the upper triangle read row by row.

In [2]:
hDD_template = ixp.declarerank2("hDD", symmetry="sym01")
independent_hDD_names = [
    str(hDD_template[i][j])
    for i in range(3)
    for j in range(i, 3)
]
expected_hDD_order = ["hDD00", "hDD01", "hDD02", "hDD11", "hDD12", "hDD22"]

print("symmetric hDD template:", hDD_template)
print("independent hDD component order:", independent_hDD_names)
if independent_hDD_names != expected_hDD_order:
    raise RuntimeError("Unexpected symmetric rank-2 component ordering.")

symmetric hDD template: [[hDD00, hDD01, hDD02], [hDD01, hDD11, hDD12], [hDD02, hDD12, hDD22]]
independent hDD component order: ['hDD00', 'hDD01', 'hDD02', 'hDD11', 'hDD12', 'hDD22']


# Step 1: Map ADM fields to BSSN variables
### [Back to [top](#Table-of-Contents)]

The BSSN formulation rewrites ADM fields into variables better suited for
numerical evolution. This notebook uses Brown's covariant BSSN idea, the
reference-metric conventions used by Baumgarte and collaborators, and the
NRPy+/reference-metric implementation linked above.

| Mathematical object | NRPy object | Role |
| --- | --- | --- |
| `gamma_ij` | `gammaDD[i][j]` | ADM spatial metric |
| `K_ij` | `KDD[i][j]` | ADM extrinsic curvature |
| `beta^i` | `betaU[i]` | ADM shift |
| `W` | `cf` when `EvolvedConformalFactor_cf = "W"` | conformal scale |
| `h_ij` | `hDD[i][j]` | conformal metric correction |
| `a_ij` | `aDD[i][j]` | trace-free curvature |
| `K` | `trK` | curvature trace |
| `vet^i` | `vetU[i]` | rescaled shift |

For the conformal metric, NRPy uses the reference metric `ghatDD` and its
determinant `detgammahat`:

$$
\bar{\gamma}_{ij}
= \left(\frac{\hat{\gamma}}{\gamma}\right)^{1/3}\gamma_{ij},
\qquad
h_{ij}
= \frac{\bar{\gamma}_{ij}-\hat{\gamma}_{ij}}{\mathrm{ReDD}_{ij}}.
$$

For the curvature variables, the conversion separates the trace from the
trace-free part:

$$
K=\gamma^{ij}K_{ij},
\qquad
\bar{A}_{ij}
= \left(\frac{\hat{\gamma}}{\gamma}\right)^{1/3}
  \left(K_{ij}-\frac{1}{3}\gamma_{ij}K\right),
\qquad
a_{ij}=\frac{\bar{A}_{ij}}{\mathrm{ReDD}_{ij}}.
$$

With the `W` conformal-factor choice,

$$
W=\left(\frac{\gamma}{\hat{\gamma}}\right)^{-1/6},
\qquad
\mathrm{vet}^i=\frac{\beta^i}{\mathrm{ReU}_i}.
$$

# Step 2: Choose the conformal-factor variable
### [Back to [top](#Table-of-Contents)]

NRPy can evolve several equivalent forms of the same metric scale. This
notebook uses `W`, the current default, because conformally flat data have the
simple expectation `cf = psi**(-2)`.

| Choice | Meaning | Conformally flat expectation |
| --- | --- | --- |
| `phi` | logarithmic scale | `log(psi)` |
| `chi` | `exp(-4 phi)` | `psi**(-4)` |
| `W` | `exp(-2 phi)` | `psi**(-2)` |

The trusted behavior is NRPy's runtime parameter interface. The new setting is
`EvolvedConformalFactor_cf`, and the output must show `W` before any
conversion result is interpreted.

In [3]:
par.set_parval_from_str("EvolvedConformalFactor_cf", "W")
cf_choice = par.parval_from_str("EvolvedConformalFactor_cf")
print("conformal factor choice:", cf_choice)
if cf_choice != "W":
    raise RuntimeError("Expected conformal factor choice W.")

conformal factor choice: W


# Step 3: Convert Brill-Lindquist ADM data
### [Back to [top](#Table-of-Contents)]

The Brill-Lindquist initial-data construction is conformally flat with zero
curvature and zero shift. The new computation below is `ADM_to_BSSN(...)` in
Cartesian coordinates. Inspect whether `cf` depends on the conformal scale;
the next validation cell checks the zero residuals.

In [4]:
with contextlib.redirect_stdout(io.StringIO()):
    brill_lindquist = InitialData_Cartesian("BrillLindquist")
    adm_to_bssn = ADM_to_BSSN(
        brill_lindquist.gammaDD,
        brill_lindquist.KDD,
        brill_lindquist.betaU,
        brill_lindquist.BU,
        "Cartesian",
    )
brill_cf_symbols = sorted(str(symbol) for symbol in adm_to_bssn.cf.free_symbols)
print("cf depends on:", brill_cf_symbols)

cf depends on: ['BH1_mass', 'BH1_posn_x', 'BH1_posn_y', 'BH1_posn_z', 'BH2_mass', 'BH2_posn_x', 'BH2_posn_y', 'BH2_posn_z', 'x', 'y', 'z']


# Validation Check: Brill-Lindquist BSSN residuals
### [Back to [top](#Table-of-Contents)]

The trusted ADM input is conformally flat, has zero curvature, and has zero
shift. The newly computed result is the BSSN variable set above. Symbol
dependence in `cf` shows where the scale went; the residuals below are the
correctness checks for `hDD`, `aDD`, `trK`, `vetU`, and `betU`.

In [5]:
brill_hDD_residuals = [
    sp.factor(adm_to_bssn.hDD[i][j])
    for i in range(3)
    for j in range(3)
]
brill_aDD_residuals = [
    sp.factor(adm_to_bssn.aDD[i][j])
    for i in range(3)
    for j in range(3)
]
brill_shift_residuals = [
    sp.factor(component)
    for component in adm_to_bssn.vetU + adm_to_bssn.betU
]
brill_trK_residual = sp.factor(adm_to_bssn.trK)

print("hDD residuals:", brill_hDD_residuals)
print("aDD residuals:", brill_aDD_residuals)
print("trK residual:", brill_trK_residual)
print("shift residuals:", brill_shift_residuals)
if (
    brill_hDD_residuals != [0] * 9
    or brill_aDD_residuals != [0] * 9
    or brill_trK_residual != 0
    or brill_shift_residuals != [0] * 6
):
    raise RuntimeError("Expected Brill-Lindquist BSSN residuals to vanish.")

hDD residuals: [0, 0, 0, 0, 0, 0, 0, 0, 0]
aDD residuals: [0, 0, 0, 0, 0, 0, 0, 0, 0]
trK residual: 0
shift residuals: [0, 0, 0, 0, 0, 0]


# Step 4: Build a toy ADM test case
### [Back to [top](#Table-of-Contents)]

The trusted test data are a conformally flat metric, symbolic curvature,
symbolic shift, and zero shift derivative:

$$
\gamma_{ij}=\psi^4\delta_{ij},
\qquad
K_{xx}=k_{xx},\ K_{xy}=K_{yx}=k_{xy},\ K_{yy}=k_{yy},\ K_{zz}=k_{zz},
\qquad
\beta^i=(\beta_0,\beta_1,\beta_2).
$$

The full 3-by-3 curvature tensor has nine slots, but symmetry leaves six
independent slots. This toy case uses four nonzero independent curvature
entries so the round-trip check exercises more than the zero-curvature
Brill-Lindquist case.

In [6]:
psi = sp.symbols("psi", positive=True)
kxx, kxy, kyy, kzz = sp.symbols("kxx kxy kyy kzz")
beta0, beta1, beta2 = sp.symbols("beta0 beta1 beta2")

toy_gammaDD = ixp.zerorank2()
toy_KDD = ixp.zerorank2()
toy_betaU = ixp.zerorank1()
toy_BU = ixp.zerorank1()
for i in range(3):
    toy_gammaDD[i][i] = psi**4
toy_KDD[0][0] = kxx
toy_KDD[0][1] = toy_KDD[1][0] = kxy
toy_KDD[1][1] = kyy
toy_KDD[2][2] = kzz
toy_betaU[0] = beta0
toy_betaU[1] = beta1
toy_betaU[2] = beta2

print("toy gammaDD diagonal:", [toy_gammaDD[i][i] for i in range(3)])
print("toy KDD nonzero entries:", [kxx, kxy, kyy, kzz])
print("toy betaU:", toy_betaU)
print("toy BU:", toy_BU)

toy gammaDD diagonal: [psi**4, psi**4, psi**4]
toy KDD nonzero entries: [kxx, kxy, kyy, kzz]
toy betaU: [beta0, beta1, beta2]
toy BU: [0, 0, 0]


The printed values are the trusted ADM input for the remaining cells.

# Step 5: Run the forward ADM-to-BSSN conversion
### [Back to [top](#Table-of-Contents)]

The new result is computed by `ADM_to_BSSN(...)`. The residuals compare it
against hand-derived conformally flat expectations for `cf`, `trK`, `hDD`, and
`vetU`. The printed `aDD[0][0]` entry is not expected to vanish; it shows that
the toy data carry nonzero trace-free curvature.

In [7]:
toy_adm_to_bssn = ADM_to_BSSN(
    toy_gammaDD,
    toy_KDD,
    toy_betaU,
    toy_BU,
    "Cartesian",
)

expected_trK = (kxx + kyy + kzz) / psi**4
cf_residual = sp.factor(toy_adm_to_bssn.cf - psi**(-2))
trK_residual = sp.factor(toy_adm_to_bssn.trK - expected_trK)
hDD_residuals = [
    sp.factor(toy_adm_to_bssn.hDD[i][j])
    for i in range(3)
    for j in range(3)
]
vetU_residuals = [
    sp.factor(toy_adm_to_bssn.vetU[i] - toy_betaU[i])
    for i in range(3)
]

print("forward cf residual:", cf_residual)
print("forward trK residual:", trK_residual)
print("forward hDD residuals:", hDD_residuals)
print("forward vetU residuals:", vetU_residuals)
print("toy aDD[0][0]:", toy_adm_to_bssn.aDD[0][0])
if (
    cf_residual != 0
    or trK_residual != 0
    or hDD_residuals != [0] * 9
    or vetU_residuals != [0, 0, 0]
):
    raise RuntimeError("Expected the forward ADM-to-BSSN residuals to vanish.")

forward cf residual: 0
forward trK residual: 0
forward hDD residuals: [0, 0, 0, 0, 0, 0, 0, 0, 0]
forward vetU residuals: [0, 0, 0]
toy aDD[0][0]: (kxx - psi**4*(kxx/psi**4 + kyy/psi**4 + kzz/psi**4)/3)/psi**4


# Step 6: Build the inverse substitution map
### [Back to [top](#Table-of-Contents)]

The zero residuals from Step 5 establish that the forward conversion matches
the hand-derived expectations checked there. `BSSN_to_ADM("Cartesian")`
builds symbolic inverse formulas containing BSSN symbols. The substitution map
below replaces each inverse-formula symbol with the value computed by
`ADM_to_BSSN`, not with expected zeros by hand.

Inspect the printed symbol list for `cf`, `trK`, the three shift symbols, and
the independent symmetric `hDD` and `aDD` symbols in the
`(00, 01, 02, 11, 12, 22)` order.

In [8]:
with contextlib.redirect_stdout(io.StringIO()):
    toy_bssn_to_adm = BSSN_to_ADM("Cartesian")
substitution_sources = []
for i in range(3):
    substitution_sources.extend(toy_bssn_to_adm.gammaDD[i])
    substitution_sources.extend(toy_bssn_to_adm.KDD[i])
substitution_sources.extend(toy_bssn_to_adm.betaU)


def forward_value_for(symbol_name):
    if symbol_name == "cf":
        return toy_adm_to_bssn.cf
    if symbol_name == "trK":
        return toy_adm_to_bssn.trK
    if symbol_name.startswith("vetU"):
        return toy_adm_to_bssn.vetU[int(symbol_name[-1])]
    if symbol_name.startswith("hDD"):
        return toy_adm_to_bssn.hDD[int(symbol_name[-2])][int(symbol_name[-1])]
    if symbol_name.startswith("aDD"):
        return toy_adm_to_bssn.aDD[int(symbol_name[-2])][int(symbol_name[-1])]
    raise ValueError(f"Unexpected BSSN symbol: {symbol_name}")


substitutions = {
    symbol: forward_value_for(symbol.name)
    for expression in substitution_sources
    for symbol in expression.free_symbols
}
substituted_symbol_names = sorted(symbol.name for symbol in substitutions)
symmetric_suffixes = ["00", "01", "02", "11", "12", "22"]
expected_substituted_symbols = (
    [f"aDD{suffix}" for suffix in symmetric_suffixes]
    + ["cf"]
    + [f"hDD{suffix}" for suffix in symmetric_suffixes]
    + ["trK", "vetU0", "vetU1", "vetU2"]
)

print("substituted BSSN symbols:", substituted_symbol_names)
if substituted_symbol_names != expected_substituted_symbols:
    raise RuntimeError("Unexpected BSSN symbols in inverse substitution map.")

substituted BSSN symbols: ['aDD00', 'aDD01', 'aDD02', 'aDD11', 'aDD12', 'aDD22', 'cf', 'hDD00', 'hDD01', 'hDD02', 'hDD11', 'hDD12', 'hDD22', 'trK', 'vetU0', 'vetU1', 'vetU2']


# Validation Check: Reconstruct ADM fields from computed BSSN values
### [Back to [top](#Table-of-Contents)]

The trusted input is the toy ADM data constructed above. The newly computed
forward result is `ADM_to_BSSN(toy data, "Cartesian")`. The inverse formulas
come from NRPy's `BSSN_to_ADM("Cartesian")`. This check substitutes the
computed BSSN values into those inverse formulas and compares every
reconstructed ADM component against the trusted toy input.

This is a symbolic consistency and integration check for this representative
toy case. It is not a proof for every possible ADM data set.

In [9]:
metric_residuals = [
    sp.factor(
        toy_bssn_to_adm.gammaDD[i][j].xreplace(substitutions) - toy_gammaDD[i][j]
    )
    for i in range(3)
    for j in range(3)
]
curvature_residuals = [
    sp.factor(toy_bssn_to_adm.KDD[i][j].xreplace(substitutions) - toy_KDD[i][j])
    for i in range(3)
    for j in range(3)
]
shift_residuals = [
    sp.factor(toy_bssn_to_adm.betaU[i].xreplace(substitutions) - toy_betaU[i])
    for i in range(3)
]
reconstructed_gammaDD = [
    [
        sp.factor(toy_bssn_to_adm.gammaDD[i][j].xreplace(substitutions))
        for j in range(3)
    ]
    for i in range(3)
]
_toy_gammaUU, toy_gamma_det = ixp.symm_matrix_inverter3x3(toy_gammaDD)
_reconstructed_gammaUU, reconstructed_gamma_det = ixp.symm_matrix_inverter3x3(
    reconstructed_gammaDD
)
determinant_residual = sp.factor(reconstructed_gamma_det - toy_gamma_det)

print("metric residuals:", metric_residuals)
print("KDD residuals:", curvature_residuals)
print("shift residuals:", shift_residuals)
print("determinant residual:", determinant_residual)
if (
    metric_residuals != [0] * 9
    or curvature_residuals != [0] * 9
    or shift_residuals != [0, 0, 0]
    or determinant_residual != 0
):
    raise RuntimeError("Expected all ADM-BSSN round-trip residuals to vanish.")

metric residuals: [0, 0, 0, 0, 0, 0, 0, 0, 0]
KDD residuals: [0, 0, 0, 0, 0, 0, 0, 0, 0]
shift residuals: [0, 0, 0]
determinant residual: 0


# What next?
### [Back to [top](#Table-of-Contents)]

The forward residuals checked the BSSN variables computed from the toy ADM
data. The substitution map then forced the inverse formulas to use those
computed values. The final residuals verified that the original ADM metric,
curvature, shift, and determinant were recovered in this representative
Cartesian case.

Reflection prompt: explain why using computed BSSN values in the substitution
map is stronger than substituting expected zeros by hand.

Next: return to the [tutorial index](../index.ipynb) to choose a generated-code
or infrastructure path.